# 또 다른 나의 세계: 평행 시나리오 생성과 감성 분석

In [ ]:
!pip install -U transformers
!pip install torch
!pip install sentencepiece
!pip install kobert-transformers
!pip install keybert
!pip install konlpy
# accelerate: "가속 및 자동 배치 도구"입니다. 모델이 GPU 메모리를 효율적으로 쓰게 해주고, device_map="auto" 같은 편리한 기능을 쓸 수 있게 해줍니다.


In [ ]:
!pip install -U huggingface_hub transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from keybert import KeyBERT
from konlpy.tag import Okt
import torch

### 평행세계(parallel_world) 이야기


**또 다른 나의 세계 만들고 평가하기**
- 스토리 생성
- 스토리 요약   
- 텍스트 분석
   - 감정 분석
   - 키워드 추출
   - 주제 분류


#### 평행세계(parallel_world) 스토리 생성

In [ ]:
!rm -rf /root/.cache/huggingface/hub/*

In [ ]:
model_name = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                              device_map='auto')
generator = pipeline("text-generation",model=model,tokenizer=tokenizer,
    max_new_tokens=200,do_sample=True,temperature=0.8,
    top_p=0.9, repetition_penalty=1.2)


In [ ]:
story = generator('나는 닌자 세계에 살고 있는 데이터분석적 닌자야, 이제 모험을 떠나려고해', max_new_tokens=300)[0]['generated_text']
story

In [ ]:
import re

clean_story = re.sub('<.*?>', '', story)
clean_story = clean_story.replace("\n", " ")
clean_story

#### 평행세계 스토리 요약

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "digit82/kobart-summarization"

# tokenizer / model 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# 토큰화
inputs = tokenizer(clean_story,return_tensors="pt",max_length=1024,truncation=True)

# 요약 생성
summary_ids = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],
    max_length=100,min_length=50,num_beams=5,no_repeat_ngram_size=3,early_stopping=True)


In [ ]:
# 평행세계 스토리 요약 디코딩
summary = tokenizer.decode(summary_ids[0],skip_special_tokens=True,clean_up_tokenization_spaces=True)

print(summary)

#### 나의 평행세계는 긍정적인가? 부정적인가?


In [ ]:
sentiment = pipeline('sentiment-analysis', model = 'beomi/KcELECTRA-base-v2022' )
senti_story = sentiment(clean_story)
print('나의 또 다른 세계의 이야기는 긍정적인가? 부정적인가? --> ', senti_story) # 약간 긍정적


#### 평행 세계의 핵심 키워드

In [ ]:
key_word_extraction = KeyBERT()
okt = Okt()

nouns = okt.nouns(clean_story)
sententce = ' '.join(nouns)
keywords = key_word_extraction.extract_keywords(sententce, top_n = 10)
print(keywords)

##또 다른 나의 세상의 모습을 확인해보자!
- ** 주의사항... 위의 모델을 학습시켜놔야함..(오래걸림..)

In [ ]:
parallel_world_sentences = [
    "나는 마법 왕국에서 데이터로 미래를 예측하는 마법사 분석가야, 이제 숨겨진 알고리즘의 탑을 향해 모험을 떠나려 해.",
    "나는 우주 제국의 데이터 항해사야, 은하의 패턴을 분석하며 새로운 별지도를 찾으러 여행을 시작하려고 해.",
    "나는 사이버 도시에서 데이터를 해킹해 진실을 찾는 분석가 요원이야, 이제 거대한 네트워크 미궁 속으로 들어가려 해.",
    "나는 고대 용들이 사는 세계에서 데이터로 전략을 짜는 전술가야, 이제 전설의 드래곤 계곡으로 탐험을 떠나려 해.",
    "나는 시간 여행자 데이터 분석가야, 역사 속 숨겨진 패턴을 찾기 위해 과거와 미래를 넘나드는 모험을 시작하려 해."
]

def parallel_world_story(start_sententce):
    # 이야기 생성
    story = generator(start_sententce)[0]['generated_text']
    clean_story = re.sub('<.*?>', '', story)
    clean_story = clean_story.replace("\n", " ")
    print('1. 나의 또 다른 세계의 이야기는 무엇일까? -->', clean_story)

    # 요약 생성
    inputs = tokenizer(clean_story,return_tensors="pt",max_length=1024,truncation=True)
    summary_ids = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],
                                max_length=100,min_length=50,num_beams=5,no_repeat_ngram_size=3,early_stopping=True)
    summary = tokenizer.decode(summary_ids[0],skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print('2. 나의 또 다른 세계의 이야기를 요약하자면? -->', summary)


    # 긍부정
    senti_clean_story = sentiment(clean_story)
    print('3. 나의 또 다른 세계의 이야기는 긍정적인가? 부정적인가? --> ', senti_story) # 약간 긍정적

    # 핵심 주제
    nouns = okt.nouns(clean_story)
    sententce = ' '.join(nouns)
    keywords = key_word_extraction.extract_keywords(sententce, top_n = 10)
    print('4. 나의 또 다른 세계의 이야기의 핵심 키워드는? --> ', keywords )
    print()
    print('*'* 200)
    print('스토리 완료')
    print('*'* 200)
    print()






In [ ]:
parallel_world_story_list = []
for another_story in parallel_world_sentences:
    story_list = parallel_world_story(another_story)
    parallel_world_story_list.append(story_list)